# Compare wavelet power based on Ra-Ap firing rate

This notebook calculates an average spike triggered wavelet spectrogram for each spike type.

## Initialization - don't change

In [1]:
import sys
sys.path.append('..')

from tqdm import tqdm

from workflows.run_util.io import collect_paths_from_table
from param_classes.workflows.wavelet_analysis import (
    WorkflowParams,
    WindowedWaveletAnalysisParams
)
from workflows.triggered_wavelet_analysis import run_analysis

params = WorkflowParams[WindowedWaveletAnalysisParams](
    analysis=WindowedWaveletAnalysisParams()
)

## Color settings

Set the colors that will be used for coloring the markers on the strip plot and the cellwise average firing frequency curve.

In [2]:
params.palette = {
    'cell_type': {
        'PYR_L2-3': '#803300ff',
        'PYR_L4-6': '#502d16ff',
        'RSI_L2-3': '#ff48a3ff',
        'RSI_L4-6': '#800080ff',
        'FSI_L2-3': '#d54500ff',
        'FSI_L4-6': '#7a0000ff'
    },
    'spike_type': {
        'RA-AP': '#00ff00',
        'UNKNOWN': '#000000b2'
    }
}

## Data settings

Set raw input dir. You need to give the base path where the group directories can be found.

In [3]:
params.data.raw_data.path='E:\\Data\\BogiSejtek\\20251007\\data\\invivo'
params.data.raw_data.extension='mat'

Set the base path of the directory where csv files containing the spike features and categories can be found.
> It needs to be the same structure as the `raw_data` directory.

In [4]:
params.data.spike_features.path = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\SpikeFeatures'
params.data.spike_features.extension = 'csv'

## Input-output settings

Set the Excel table containing the cell attributes, grouping and selector information.

In [5]:
params.input = {
    'mode': 'table',
    'table_path': 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\data_invivo.xlsx',
    'select_rules': [
        {'selected': 1}
    ]
}

Set the output base directory path. All results of this analysis (cellwise aggregated tables, grouped results, figures, statistics) is going to saved there.

In [6]:
params.output.path = 'E:\\Data\\BogiSejtek\\20251007\\data\\invivo\\Preprocessed\\Power_RaAp_Firing_Relationship'

## Analysis parameters

 - `params.analysis.spike_types`: the list of the spike types to use as trigger

In [7]:
params.analysis.spike_types = ['AIS-AP', 'UNKNOWN', 'RA-AP']

 - `params.analysis.cut_window`: window size to cut, relative to the spike

In [8]:
params.analysis.window_size = 1.0
params.analysis.step_size = 0.1

 - `params.analysis.signal.name`: signal name to analyze
 - `params.analysis.signal.fs`: resampled sampling rate of the signal

In [9]:
params.analysis.signal.name = 'wideband'
params.analysis.signal.fs = 5000

 - `params.analysis.wavelet.low`: lowest analyzed frequency
 - `params.analysis.wavelet.high`: highest analyzed frequency
 - `params.analysis.wavelet.step`: step size between analyzed frequencies
 - `params.analysis.wavelet.n_cycles`: number of cycles in the wavelet
 - `params.analysis.wavelet.length`: length of the wavelet

In [10]:
params.analysis.wavelet.low = 1.0
params.analysis.wavelet.high = 30.0
params.analysis.wavelet.step = 1.0
params.analysis.wavelet.n_cycles = 7
params.analysis.wavelet.length = 1.0

In [11]:
raap_threshold = 10

## Plot parameters

 - `params.analysis.spectrogram.xlim`: limit of the x axis in seconds

In [12]:
params.analysis.xlim = (-3, 0.5)

 - `params.analysis.spectrogram.ylim`: limit of the y axis

In [13]:
params.analysis.ylim = (0, 30)

 - `params.analysis.spectrogram.clim`: ends of the colorbar

In [14]:
params.analysis.clim = (None, None)

## Initialize analysis

In [15]:
from workflows.windowed_wavelet_analysis import (
    create_recording_wavelet_calculator,
    create_wavelet_saver,
    create_windowed_segmenter,
    aggregate_spectrograms,
    save_spectrogram_figure,
    plot_spectrogram_with_power
)
process_cell = create_recording_wavelet_calculator(params)
save_spectrogram = create_wavelet_saver(params.output.path)
create_segments = create_windowed_segmenter(params.analysis)

## Run analysis

Collect input files according to the table.

In [16]:
input_files = collect_paths_from_table(params.data, params.input)

### Individual analysis

Perform individual analysis and save the results.

In [17]:
import numpy as np
import matplotlib.pyplot as plt

from perform_full_wavelet import Spectrogram

for cell in tqdm(input_files, total=len(input_files), desc="Processing cells"):
    print(f"Processing cell: {cell.info.name}")
    cell_data = cell.data.load()
    spectrogram = process_cell(cell_data)
    segments = create_segments(spectrogram, cell_data.spikes)
    baseline_segments, raap_segments = [], []
    for seg in segments:
        if seg.spike_counts['RA-AP'] == 0:
            baseline_segments.append(seg)
        else:
            raap_segments.append(seg)
    raap_freqs = [
        seg.get_firing_rate('RA-AP') for seg in raap_segments
    ]
    edges = np.quantile(raap_freqs, np.arange(0, 1.1, 0.2))
    baseline_spectrogram = aggregate_spectrograms(
        [seg.spectrogram for seg in baseline_segments]
    )
    raap_active_segments = [
        seg for seg in raap_segments
        if seg.get_firing_rate('RA-AP') >= raap_threshold
    ]
    baseline_power = np.median(baseline_spectrogram.power, axis=1)
    baseline_spectrogram = baseline_spectrogram.with_power(
        (baseline_spectrogram.power / baseline_power.reshape(-1, 1))
    )
    baseline_figure = plot_spectrogram_with_power(
        baseline_spectrogram,
        clim=params.analysis.clim,
        color='blue',
        fig_size=(5, 3)
    )
    save_spectrogram_figure(
        baseline_figure,
        f"{cell.info.name}_baseline_normalized_spectrogram",
        params.output.path,
    )
    plt.close(baseline_figure)
    for i, segment in enumerate(raap_active_segments):
        spectrogram: Spectrogram = segment.spectrogram
        spectrogram = spectrogram.with_power(
            (spectrogram.power / baseline_power.reshape(-1, 1))
        )
        segment_figure = plot_spectrogram_with_power(
            spectrogram,
            clim=params.analysis.clim,
            color='red',
            fig_size=(5, 3)
        )
        segment_figure.suptitle(
            f"{cell.info.name} | RA-AP: {segment.get_firing_rate('RA-AP'):.2f} Hz"
        )
        save_spectrogram_figure(
            segment_figure,
            f"{cell.info.name}_RaAp_active_segment_{i:002d}",
            params.output.path,
        )
        plt.close(segment_figure)
    break
    # save_spectrogram(spectrogram, cell.info.name)

Processing cells:   0%|          | 0/3 [00:00<?, ?it/s]

Processing cell: B18_d3t1c1


Processing cells:   0%|          | 0/3 [01:07<?, ?it/s]


Sort cells into groups by the `group` column of the Excel table.

In [18]:
# group_names = list(set([file.info.cell_type for file in input_files]))
# groups = []
# for group_name in group_names:
#     group_files = [
#         file for file in cell_results
#         if file.info.cell_type == group_name
#     ]
#     groups.append((group_name, group_files))
# print(f"{len(groups)} groups found for aggregation:")
# for group in groups:
#     print(f"\t- {group[0]}")

Aggregate each cells within groups. It will
 - average triggered histograms that satisfies the condition
 - detect maximum time and value
 - plot and save the aggregation result

In [19]:
# group_aggregates = []
# for group_name, group_files in groups:
#     group_aggregates.append(run_grouped_analysis(group_files, group_name, params))

### Grouped analysis

In [20]:
# from workflows.histogram_grouping import run_grouped_analysis
# params.grouping.flag = 1
# params.grouping.path = 'C:\\tmp\\Bozso2025\\PutRAAP\\Results\\New\\data_invivo.xlsx'
# params.grouping.column_name = 'group'
# params.grouping.groups =  ['sw']
# params.grouping.spike_types = ['AIS-AP', 'RA-AP']
# params.grouping.conditions = ['RA-AP']

# run_grouped_analysis(
#     config=params.grouping,
#     cell_results=cell_results
# )